In [1]:
!pip install transformers==5.3.0 tslearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.9/387.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.9 MB/s eta 0:00:00
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: numba
    Found existing installation: numba 0.60.0
    Uninstalling numba-0.60.0:
      Successfully uninstalled numba-0.60.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 re

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, TimesFm2_5ModelForPrediction, AutoModel

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, balanced_accuracy_score

import numpy as np
from tslearn.datasets import UCR_UEA_datasets

def load_lsst_data():
    """Load the LSST dataset and format dimensions."""
    # Load the LSST dataset from UEA archive
    ds = UCR_UEA_datasets()
    X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

    # Swap axes to match (n_samples, n_channels, n_timesteps) format
    X_train = np.swapaxes(X_train, 1, 2)
    X_test = np.swapaxes(X_test, 1, 2)

    # Flatten labels for scikit-learn
    y_train = y_train.ravel()
    y_test = y_test.ravel()

    return X_train, y_train, X_test, y_test

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load LSST dataset
X_train, y_train, X_test, y_test = load_lsst_data()

# Encode labels to integers
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
num_classes = len(label_encoder.classes_)
print(num_classes)

# ---------------------------------------------------------
# STANDARD SCALING FOR 3D TIME SERIES
# ---------------------------------------------------------
samples_train, channels, timesteps = X_train.shape
samples_test = X_test.shape[0]

# Reshape from (samples, channels, timesteps) to (samples * timesteps, channels)
# This format allows the scaler to compute 1 mean and 1 std per channel
X_train_reshaped = X_train.transpose(0, 2, 1).reshape(-1, channels)
X_test_reshaped = X_test.transpose(0, 2, 1).reshape(-1, channels)

scaler = StandardScaler()

# Fit ONLY on the training data, then transform both sets
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_test_scaled = scaler.transform(X_test_reshaped)

# Reshape back to the original 3D shape (samples, channels, timesteps)
X_train = X_train_scaled.reshape(samples_train, timesteps, channels).transpose(0, 2, 1)
X_test = X_test_scaled.reshape(samples_test, timesteps, channels).transpose(0, 2, 1)
# ---------------------------------------------------------

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True) # Reduced batch size for memory
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

14


In [4]:
import torch.nn.functional as F

class TimesFMClassifier(nn.Module):
    def __init__(self, model_name="google/timesfm-2.5-200m-transformers", num_classes=14):
        super().__init__()

        self.foundation_model = AutoModel.from_pretrained(model_name)

        for param in self.foundation_model.parameters():
            param.requires_grad = False

        hidden_size = self.foundation_model.config.hidden_size
        self.classifier = nn.Sequential(
            # This mathematically acts like a StandardScaler dynamically on the embeddings
            nn.BatchNorm1d(num_features=hidden_size),
            nn.Linear(in_features=hidden_size, out_features=num_classes)
        )

    def forward(self, x):
        batch_size, n_channels, n_timesteps = x.shape
        x_reshaped = x.view(batch_size * n_channels, n_timesteps)

        # FIX: Dynamically pad the sequence length to be a multiple of the model's patch length (32)
        patch_len = 32
        remainder = n_timesteps % patch_len
        if remainder != 0:
            pad_amount = patch_len - remainder
            # Pad the last dimension (time) on the right side with zeros
            x_reshaped = F.pad(x_reshaped, (0, pad_amount), "constant", 0)

        with torch.no_grad():
            outputs = self.foundation_model(past_values=x_reshaped)
            hidden_states = outputs.last_hidden_state
            pooled_embeddings = hidden_states.mean(dim=1)

        pooled_embeddings = pooled_embeddings.view(batch_size, n_channels, -1)
        final_embedding = pooled_embeddings.mean(dim=1)

        logits = self.classifier(final_embedding)
        return logits
model = TimesFMClassifier()
print(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

TimesFm2_5Model LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
output_projection_point.residual_layer.weight     | UNEXPECTED |  | 
output_projection_point.output_layer.weight       | UNEXPECTED |  | 
output_projection_point.input_layer.weight        | UNEXPECTED |  | 
output_projection_quantiles.residual_layer.weight | UNEXPECTED |  | 
output_projection_quantiles.input_layer.weight    | UNEXPECTED |  | 
output_projection_quantiles.output_layer.weight   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TimesFMClassifier(
  (foundation_model): TimesFm2_5Model(
    (input_ff_layer): TimesFm2_5ResidualBlock(
      (input_layer): Linear(in_features=64, out_features=1280, bias=True)
      (activation): SiLU()
      (output_layer): Linear(in_features=1280, out_features=1280, bias=True)
      (residual_layer): Linear(in_features=64, out_features=1280, bias=True)
    )
    (layers): ModuleList(
      (0-19): 20 x TimesFm2_5DecoderLayer(
        (self_attn): TimesFm2_5Attention(
          (q_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (v_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (o_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (q_norm): TimesFm2_5RMSNorm((80,), eps=1e-06)
          (k_norm): TimesFm2_5RMSNorm((80,), eps=1e-06)
        )
        (mlp): TimesFm2_5MLP(
          (activation_fn): SiLU()
          (fc1): Linear(in_featur

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report # Added classification_report

config = {
    "learning_rate": 1e-3,
    "epochs": 12,
    "batch_size": 32,
    "architecture": "TimesFM-Adapter",
    "dataset": "LSST"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print('yes')
criterion = nn.CrossEntropyLoss()
# Pass only the newly initialized classifier parameters to the optimizer
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
print('begin training')

for epoch in range(config["epochs"]):
    # --- Training Phase ---
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # --- Evaluation Phase ---
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_targets.extend(batch_y.numpy())

    # Calculate robust metrics for imbalanced classes
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    bal_acc = balanced_accuracy_score(all_targets, all_preds)

    print(f"Epoch {epoch+1}/{config['epochs']} | Loss: {avg_train_loss:.4f} | Macro F1: {macro_f1:.4f} | Bal Acc: {bal_acc:.4f}")

# --- Final Classification Report ---
print("\n" + "="*50)
print("FINAL CLASSIFICATION REPORT (Test Set)")
print("="*50)
# If you have your label_encoder available, you can pass target_names to see the actual class names:
# target_names = [str(cls) for cls in label_encoder.classes_]
# print(classification_report(all_targets, all_preds, target_names=target_names))

# Otherwise, printing it directly will show the integer class labels:
print(classification_report(all_targets, all_preds))